### 卷积神经网络（converlutional Neural Network）


### 关于这个Notebook

在这个Notebook中，我们将学习如何使用pytorch框架来构建简单的卷积神经网络，以及使用该网络对自己采集的表情数据集进行识别。
首先，我们会手动搭建卷积神经网络，调整超参数，进行图像分类训练。

最后，我们可以尝试着使用torch内置的网络（AlexNet/Resnet），对数据集进行迁移学习。

相比于上一节的手写数字识别，这一次我们的不同点是：1、我们需要用卷积神经网络进行图片分类。 2、我们用的是自己的数据集，因此需要自己构建自己的dataset， dataloader 类。

然后，我们结合OpenCV课程中用到的人脸检测模型（haar+adaboost/已经训练好的pb模型），先对人脸进行检测，再对表情进行识别，达到多模型融合推理的效果。

In [1]:
import os
import torch
import sys
from PIL import Image
import torchvision
import matplotlib
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset
from torchvision import transforms
from tqdm import tqdm
import torch.optim as optim

#### 采集数据集

In [ ]:
## import numpy as np
import cv2
import time
import os
from PIL import Image

cap = cv2.VideoCapture(0)
cap.set(3, 640)
cap.set(4, 480)

font                   = cv2.FONT_HERSHEY_SIMPLEX
bottomLeftCornerOfText = (10, 25)
fontScale              = 1
fontColor              = (255,255,255)
lineType               = 2

print('Input image class index: ', end = '')
class_index = input()

print('Input maximum image number of class {}: '.format(class_index), end = '')
max_image_num = input()

model_bin = "./models/models/face_detector/opencv_face_detector_uint8.pb"
config_text = "./models/models/face_detector/opencv_face_detector.pbtxt";

net_face = cv2.dnn.readNetFromTensorflow(model_bin, config=config_text)


folder_path = os.path.join('./selfpick/validation', str(class_index))
if not os.path.exists(folder_path):
    os.makedirs(folder_path)

i = 0
while(True):
    ret, frame = cap.read()
    h, w, c = frame.shape
    blobImage = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), (104.0, 177.0, 123.0), False, False);
    net_face.setInput(blobImage)
    cvOut = net_face.forward()
   # 检测人脸位置
    for detection in cvOut[0,0,:,:]:
        score = float(detection[2])
        objIndex = int(detection[1])
        if score > 0.5:
            left = int(detection[3]*w)
            top = int(detection[4]*h)
            right = int(detection[5]*w)
            bottom = int(detection[6]*h)
        
            saved_file_path = os.path.join(folder_path, 'image_' + str(i) + '.png')
            pic = frame[top:bottom,left:right]
            resized = cv2.resize(pic,(240,320))
            cv2.imwrite(saved_file_path,resized)
            img = cv2.rectangle(frame,(left, top), (right, bottom),(255, 0, 0), 2)

            cv2.putText(frame,'Image: ' + str(i), 
                bottomLeftCornerOfText, 
                font, 
                fontScale,
                fontColor,
                lineType)

    cv2.imshow('frame', frame)
    cv2.waitKey(1)
    i+=1

    if cv2.waitKey(1) == 27 or i >= int(max_image_num):
        break

cap.release()
cv2.destroyAllWindows()

Input image class index: 

### 创建自己的DATASET类

In [2]:
class MyDataSet(Dataset):
    """自定义数据集"""

    def __init__(self, images_path: list, images_class: list, transform=None):
        self.images_path = images_path
        self.images_class = images_class
        self.transform = transform

    def __len__(self):
        return len(self.images_path)

    def __getitem__(self, item):
        img = Image.open(self.images_path[item])
#         print(img.shape)
        # RGB为彩色图片，L为灰度图片
        if img.mode != 'RGB':
            raise ValueError("image: {} isn't RGB mode.".format(self.images_path[item]))
        label = self.images_class[item]

        if self.transform is not None:
            img = self.transform(img)
        return img, label
    
    @staticmethod
    def collate_fn(batch):
        #自定义数据加载和批处理的方式
        # 官方实现的default_collate可以参考
        # https://github.com/pytorch/pytorch/blob/67b7e751e6b5931a9f45274653f4f653a4e6cdf6/torch/utils/data/_utils/collate.py
        images, labels = tuple(zip(*batch)) #将列表解包为两个独立元组：images = (img1, img2, ..., imgN) labels = (label1, label2, ..., labelN)

        images = torch.stack(images, dim=0) #在维度 dim=0（批次维度）堆叠，生成形状为 [N, C, H, W] 的张量
        labels = torch.as_tensor(labels)
        return images, labels

        

In [3]:
def create_dataset(root):  #这里的root是数据集根目录的路径，注意分类数据集的数据存放形式
    all_class = [cla for cla in os.listdir(root) if os.path.isdir(os.path.join(root, cla))]  #读取所有类别
    all_class.sort()
    class_indices = dict((k, v) for v, k in enumerate(all_class)) # 为每个类别分配一个数字
    images_path = []
    images_label = []
    for cla in all_class:
        cla_path = os.path.join(root, cla)
        images = [os.path.join(root, cla, i) for i in os.listdir(cla_path)] #读取所有图片的完整相对路径
        image_class = class_indices[cla] #获取label
        for image_path in images:
            images_path.append(image_path)
            images_label.append(image_class)
        

    return images_path, images_label

root = "selfpick"
train_images_path, train_images_label = create_dataset(os.path.join(root, 'train'))
val_images_path, val_images_label = create_dataset(os.path.join(root, 'validation')) 

transform=transforms.Compose([
        transforms.ToTensor()]) 

train_dataset = MyDataSet(train_images_path, train_images_label, transform)
val_dataset = MyDataSet(val_images_path, val_images_label, transform)

### 创建自己的dataloader

In [4]:
batch_size = 32
nw = min([os.cpu_count(), batch_size if batch_size > 1 else 0, 8]) # 加载数据集所需worker
train_loader = torch.utils.data.DataLoader(train_dataset,
                                               batch_size=batch_size,
                                               shuffle=True
                                               )

val_loader = torch.utils.data.DataLoader(val_dataset,
                                             batch_size=batch_size,
                                             shuffle=False
                                             )

In [5]:
#查看计算平台 看是否有Gpu可用
device = torch.device('cuda:0' if torch.cuda.is_available() else "cpu")
print(device)

cuda:0


#### pytorch搭建卷积神经网络模型

首先搭建一个六层的卷积神经网络，4层卷积层，2层全连接。

输入为(240,320)彩色图，第一个层卷积层一个kernel/filter 的shape是3x3，共32个kernel/filter组成，relu函数作为激活函数，有padding。

接一个2x2的Maxpooling层.

第二个卷积层kernel3x3，共64个kernel/filter组成，relu函数作为激活函数，padding。

接一个2x2的Maxpooling层.

第三个卷积层kernel 3x3，共128个kernel/filter组成，relu函数作为激活函数，padding。

接一个2x2的Maxpooling层.

第四个卷积层kernel 3x3，共256个kernel/filter组成，relu函数作为激活函数，padding。

接一个2x2的Maxpooling层.


然后拉成一维，再跟100个神经元做一个全连接，relu函数作为激活函数。

最后，输出层3个神经元（多分类问题），softmax作为激活函数。

In [ ]:
def conv_block(in_channels, out_channels, pool_kernel, pool_stride):
    """构建一个卷积块：卷积 -> BN -> ReLU -> 池化"""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
        nn.BatchNorm2d(out_channels),
        nn.ReLU(),
        nn.MaxPool2d(kernel_size=pool_kernel, stride=pool_stride)
    )

class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            conv_block(3, 32, pool_kernel=2, pool_stride=2),
            conv_block(32, 64, pool_kernel=2, pool_stride=2),
            conv_block(64, 128, pool_kernel=4, pool_stride=4),
            conv_block(128, 256, pool_kernel=4, pool_stride=4)
        )
        self.classifier = nn.Sequential(
            nn.Linear(3840, 100),
            nn.ReLU(),
            nn.Linear(100, 3)
        )

    def forward(self, x):
        x = self.features(x)
        # 将特征图展平成一维向量
        x = x.view(x.size(0), -1)
        
        x = self.classifier(x)
        return x
net = CNN().to(device)

**想一想：你会计算params的个数吗？**

### 训练模型
我们先定义一下训练一个epoch函数和评价的函数
model.train() 表示参数可被修改

model.eval() 表示参数不被修改

In [7]:
model = net.to(device)
optimizer = optim.AdamW(model.parameters(), lr=0.001)
def train_one_epoch(model, optimizer, data_loader, device, epoch):
    model.train()
    loss_function = torch.nn.CrossEntropyLoss()
    accu_loss = torch.zeros(1).to(device)  # 累计损失
    accu_num = torch.zeros(1).to(device)   # 累计预测正确的样本数
    optimizer.zero_grad()

    sample_num = 0
    data_loader = tqdm(data_loader, file=sys.stdout)
    for step, data in enumerate(data_loader):
        images, labels = data
        print(images.shape)
        sample_num += images.shape[0]

        pred = model(images.to(device))
        #print(pred)
        pred_classes = torch.max(pred, dim=1)[1]
        #print(pred_classes)
        accu_num += torch.eq(pred_classes, labels.to(device)).sum()
        #print(accu_num)
        loss = loss_function(pred, labels.to(device))
        loss.backward()
        accu_loss += loss.detach()
        print(accu_loss)

        data_loader.desc = "[train epoch {}] loss: {:.3f}, acc: {:.3f}".format(epoch,
                                                                               accu_loss.item() / (step + 1),
                                                                               accu_num.item() / sample_num)

        if not torch.isfinite(loss):
            print('WARNING: non-finite loss, ending training ', loss)
            sys.exit(1)

        optimizer.step()
        optimizer.zero_grad()

    return accu_loss.item() / (step + 1), accu_num.item() / sample_num


@torch.no_grad()
def evaluate(model, data_loader, device, epoch):
    loss_function = torch.nn.CrossEntropyLoss()

    model.eval()

    accu_num = torch.zeros(1).to(device)   # 累计预测正确的样本数
    accu_loss = torch.zeros(1).to(device)  # 累计损失

    sample_num = 0
    data_loader = tqdm(data_loader, file=sys.stdout)
    for step, data in enumerate(data_loader):
        images, labels = data
        sample_num += images.shape[0]

        pred = model(images.to(device))
        pred_classes = torch.max(pred, dim=1)[1]
        accu_num += torch.eq(pred_classes, labels.to(device)).sum()

        loss = loss_function(pred, labels.to(device))
        accu_loss += loss

        data_loader.desc = "[valid epoch {}] loss: {:.3f}, acc: {:.3f}".format(epoch,
                                                                               accu_loss.item() / (step+1),
                                                                               accu_num.item() / sample_num)

    return accu_loss.item() / (step + 1), accu_num.item() / sample_num


### 开始训练
每10轮保存一次模型

In [8]:
epochs = 5
for epoch in range(epochs):
        # train
        train_loss, train_acc = train_one_epoch(model=model,
                                                optimizer=optimizer,
                                                data_loader=train_loader,
                                                device=device,
                                                epoch=epoch)

        # validate
        val_loss, val_acc = evaluate(model=model,
                                     data_loader=val_loader,
                                     device=device,
                                     epoch=epoch)
        

  0%|          | 0/29 [00:00<?, ?it/s]

torch.Size([32, 3, 320, 240])
torch.Size([32, 3840]) 32 32
tensor([1.0737], device='cuda:0')
[train epoch 0] loss: 1.074, acc: 0.406:   3%|▎         | 1/29 [00:35<16:37, 35.62s/it]torch.Size([32, 3, 320, 240])
torch.Size([32, 3840]) 32 32
tensor([3.8756], device='cuda:0')
[train epoch 0] loss: 1.938, acc: 0.422:   7%|▋         | 2/29 [00:36<06:47, 15.08s/it]torch.Size([32, 3, 320, 240])
torch.Size([32, 3840]) 32 32
tensor([6.7892], device='cuda:0')
[train epoch 0] loss: 2.263, acc: 0.406:  10%|█         | 3/29 [00:37<03:41,  8.53s/it]torch.Size([32, 3, 320, 240])
torch.Size([32, 3840]) 32 32
tensor([8.7224], device='cuda:0')
[train epoch 0] loss: 2.181, acc: 0.398:  14%|█▍        | 4/29 [00:37<02:15,  5.43s/it]torch.Size([32, 3, 320, 240])
torch.Size([32, 3840]) 32 32
tensor([10.4720], device='cuda:0')
[train epoch 0] loss: 2.094, acc: 0.388:  17%|█▋        | 5/29 [00:38<01:29,  3.74s/it]torch.Size([32, 3, 320, 240])
torch.Size([32, 3840]) 32 32
tensor([11.5691], device='cuda:0')
[trai

In [ ]:
#torch.save(model, "./model.pth")

torch.save(model.state_dict(), "./model.pth")

### 用Opencv读取人脸检测、表情识别pt模型并推理
在OpenCV那一节当中，我们学了如何使用dnn中的推理模块读取人脸检测模型，进行人脸检测，画出人脸框。训练了我们自己的人脸表情识别模型后，我们可以将其加入，在检测好人脸之后，在人脸区域运用表情识别推理出人的表情，并显示。

In [ ]:
import cv2
import numpy as np
import time

model_bin = "./models/models/face_detector/opencv_face_detector_uint8.pb"
config_text = "./models/models/face_detector/opencv_face_detector.pbtxt";

class_names = ['cry','happy','normal']

#加载训练好的模型
#net = torch.load('model.pth')
net = CNN()
net.load_state_dict(torch.load('model.pth'))
# model.eval()

C:\Users\ninil\AppData\Local\Temp\ipykernel_25424\2900805092.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  net.load_state_dict(torch.load('model.pth'))


<All keys matched successfully>

In [10]:
def get_face_data(): 
    # 加载模型文件，用于检测人脸
    net_face = cv2.dnn.readNetFromTensorflow(model_bin, config=config_text)
    
    
    font = cv2.FONT_HERSHEY_SIMPLEX
    
    camera = cv2.VideoCapture(0)
    count = 1
    while True:
        ret, frame = camera.read()
        h, w, c = frame.shape
        blobImage = cv2.dnn.blobFromImage(frame, 1.0, (300, 300), (104.0, 177.0, 123.0), False, False);
        net_face.setInput(blobImage)
        cvOut = net_face.forward()
       # 检测人脸位置
        for detection in cvOut[0,0,:,:]:
            score = float(detection[2])
            objIndex = int(detection[1])
            if score > 0.5:
                left = int(detection[3]*w)
                top = int(detection[4]*h)
                right = int(detection[5]*w)
                bottom = int(detection[6]*h)
             
                img = cv2.rectangle(frame,(left, top), (right, bottom),(255, 0, 0), 2)
                start = time.time()
                # 根据检测结果截取图片并调整截取后的图片大小，为进一步做表情识别做准备
                ### start code
                f = frame[top:bottom, left:right]
                
                ##将检测到的人脸resize成跟训练集图像同样大小
                f = cv2.resize(frame[top:bottom, left:right], (240, 320))
                ##将图像变换成tensor
                print(f.shape)
                ##将图像由[c,h,w] 的tensor 变换成[b,c,h,w]的tensor 以符合dataloader训练集的shape
                img_tensor = transform(f)
                #img_tensor = trans(f)
                print(img_tensor.shape)
                #img_tensor.to(device)
                img_tensor = img_tensor.unsqueeze(0)
                print(img_tensor.shape)
                  
                ## 进行推理
                prediction = net(img_tensor)
                ## 将推理结果取回cpu
                ### end code
                prediction = prediction.detach()
                end = time.time()
                fps = 1/(end - start)
                fps_label = "Throughput: %.2f FPS" % fps
                cv2.putText(frame, fps_label, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)
                
                print(np.array(prediction))
                idx = np.argmax(np.array(prediction))

                result = class_names[idx]
                cv2.putText(frame, result, (left,top), font, 2, (255, 0, 0), 2, cv2.LINE_AA)
                count += 1
             
        cv2.imshow('pic', frame)
        # 停止程序
        if cv2.waitKey(10) & 0xff == ord(' '):
            break
    camera.release()
    cv2.destroyAllWindows()
 
if __name__=='__main__':
    get_face_data()

(320, 240, 3)
torch.Size([3, 320, 240])
torch.Size([1, 3, 320, 240])
torch.Size([1, 3840]) 1 1
[[-6.1733084  -6.813593   -0.54717076]]
(320, 240, 3)
torch.Size([3, 320, 240])
torch.Size([1, 3, 320, 240])
torch.Size([1, 3840]) 1 1
[[-7.711221   -5.1070495  -0.34309193]]
(320, 240, 3)
torch.Size([3, 320, 240])
torch.Size([1, 3, 320, 240])
torch.Size([1, 3840]) 1 1
[[-5.4624023  -0.06115742 -2.720039  ]]
(320, 240, 3)
torch.Size([3, 320, 240])
torch.Size([1, 3, 320, 240])
torch.Size([1, 3840]) 1 1
[[-6.177979  -1.732399  -1.7970958]]
(320, 240, 3)
torch.Size([3, 320, 240])
torch.Size([1, 3, 320, 240])
torch.Size([1, 3840]) 1 1
[[-6.0145345 -2.2447941 -2.091077 ]]
(320, 240, 3)
torch.Size([3, 320, 240])
torch.Size([1, 3, 320, 240])
torch.Size([1, 3840]) 1 1
[[-6.530412  -2.8256958 -1.8755112]]
(320, 240, 3)
torch.Size([3, 320, 240])
torch.Size([1, 3, 320, 240])
torch.Size([1, 3840]) 1 1
[[-5.8043494 -3.1610014 -2.2041712]]
(320, 240, 3)
torch.Size([3, 320, 240])
torch.Size([1, 3, 320, 240]

### torch中自带网络模型
如果我们不自己搭建卷积神经网络，也可以直接导入一些成熟的网络模型，Torch中的models包中提供了带有预训练权值的深度学习模型，这些模型可以用来进行预测、特征提取和微调（fine-tuning).
>Xception
>VGG16
>VGG19
>ResNet, ResNetV2, ResNeXt
>InceptionV3
>InceptionResNetV2
>MobileNet
>MobileNetV2
>DenseNet
>NASNet

In [ ]:
net_res = torchvision.models.resnet18(pretrained = True)# 这里的True意味着是经过官方预训练的,模型收敛速度要快的多得多
fc_inputs=net_res.fc.in_features
net_res.fc= nn.Linear(fc_inputs,3) #这里的3需要改成你自己数据集的需要分类的种类

C:\Users\ninil\AppData\Roaming\Python\Python39\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\ninil\AppData\Roaming\Python\Python39\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


In [ ]:
model_res = net_res.to(device)
pg = [p for p in model_res.parameters() if p.requires_grad]
optimizer = optim.AdamW(pg, lr=0.001, weight_decay=5E-2)

In [ ]:
epochs = 2
for epoch in range(epochs+1):
        # train
        train_loss, train_acc = train_one_epoch(model=model_res,
                                                optimizer=optimizer,
                                                data_loader=train_loader,
                                                device=device,
                                                epoch=epoch)

        # validate
        val_loss, val_acc = evaluate(model=model_res,
                                     data_loader=val_loader,
                                     device=device,
                                     epoch=epoch)


  0%|          | 0/29 [00:00<?, ?it/s]torch.Size([32, 3, 320, 240])
tensor([1.2201], device='cuda:0')
[train epoch 0] loss: 1.220, acc: 0.375:   3%|▎         | 1/29 [00:00<00:22,  1.26it/s]torch.Size([32, 3, 320, 240])
tensor([1.4434], device='cuda:0')
[train epoch 0] loss: 0.722, acc: 0.656:   7%|▋         | 2/29 [00:01<00:14,  1.88it/s]torch.Size([32, 3, 320, 240])
tensor([1.7466], device='cuda:0')
[train epoch 0] loss: 0.582, acc: 0.750:  10%|█         | 3/29 [00:01<00:11,  2.25it/s]torch.Size([32, 3, 320, 240])
tensor([1.9127], device='cuda:0')
[train epoch 0] loss: 0.478, acc: 0.789:  14%|█▍        | 4/29 [00:01<00:09,  2.52it/s]torch.Size([32, 3, 320, 240])
tensor([2.1551], device='cuda:0')
[train epoch 0] loss: 0.431, acc: 0.819:  17%|█▋        | 5/29 [00:02<00:08,  2.69it/s]torch.Size([32, 3, 320, 240])
tensor([2.7396], device='cuda:0')
[train epoch 0] loss: 0.457, acc: 0.833:  21%|██        | 6/29 [00:02<00:08,  2.80it/s]torch.Size([32, 3, 320, 240])
tensor([2.8283], device='c

In [ ]:
torch.save(model_res.state_dict(), "./model-res.pth")
#torch.save(model_res,"./resmodel.pth")

In [ ]:
#net_res = torch.load('./model-res.pth')
net_res.load_state_dict(torch.load('./model-res.pth')) 

C:\Users\ninil\AppData\Local\Temp\ipykernel_33068\3376724154.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  net_res.load_state_dict(torch.load('./model-res.pth'))


<All keys matched successfully>